In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp
import sys
sys.path.insert(0, '/cvmfs/larsoft.opensciencegrid.org/spack-fnal-v1.0.0/opt/spack/linux-x86_64_v2/root-6.28.12-vgs6mjswsg36hl3oarsrsyc2dcua6khe/lib/root')

import ROOT

import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd
import gc

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *
from analysis_village.cc1pi.var_configs import *



from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks import CutMasks
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

from analysis_village.cc1pi.Optimize import OptimizationUtils
from analysis_village.cc1pi.Optimize import ConfusionMatricesUtils

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *


# Load DataFrames

In [ ]:
## Check keys in each file
optimization_file = "/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_1e20_training_update_cv.df"
#optimization_file = "/exp/sbnd/data/users/lpelegri/cafpyana_data/wrong_energy_cut/old2/cc1pi_1e20_training.df"
splh.print_keys(optimization_file)
## Check keys in each file

SLICE_LEVELS = ["__ntuple", "entry", "rec.slc..index"]

cols_to_keep = [
('nu_categ', '', '', ''),
('nu_categ_proton_reduced', '', '', ''),
('genie_categ', '', '', ''),
('genie_mode', '', '', ''),
]
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file

pot_weight_col = ('slc', 'wgt', '', '', '', '')


In [ ]:
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file

optimization_df = load_df(optimization_file, keys2load, 100, filter_df = True, reprocess_df = True, reprocess_truth = True)
optimization_evt_df = optimization_df['cc1pi']
optimization_hdr_df = optimization_df['hdr']
optimization_nu_df = optimization_df['nudf']

print("data_tot_pot: %.3e" %(data_tot_pot))

keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
optimization_tot_pot = optimization_hdr_df['pot'].sum()
mc_pot_scale = data_tot_pot / optimization_tot_pot
print("mc_tot_pot: %.3e" %(optimization_tot_pot))
print("mc_pot_scale: %.3e" %(mc_pot_scale))
optimization_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(optimization_evt_df))

optimization_evt_df = perform_truth_matching(optimization_evt_df, optimization_nu_df[cols_to_keep])

evt_df = optimization_evt_df

# BC Flash matcher

In [ ]:
file_dir = "/exp/sbnd/data/users/lpelegri/Graphs/Optimization"
os.makedirs(file_dir, exist_ok=True)  # create directory if needed

In [ ]:
# Drop the last index level (rec.slc.reco.pfp..index)

mask = (evt_df.slc.cut.obvious_cosmic == True) &  (evt_df.slc.cut.inside_FV == True) 
evt_df_reset = evt_df[mask].reset_index(level='rec.slc.reco.pfp..index', drop=True)

# Drop duplicates based on the remaining index levels
evt_df_unique = evt_df_reset[~evt_df_reset.index.duplicated(keep='first')]

# Split signal and background
signal_df = evt_df_unique[(evt_df_unique.truth.nu_categ != "cosmic")]
bkg_df    = evt_df_unique[(evt_df_unique.truth.nu_categ == "cosmic")]

column = ('slc', 'barycenterFM', 'score', '', '', '')

df_opt, best, fig = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column=('slc', 'barycenterFM', 'score', '', '', ''),
    cut_type=">",
    xlabel="Barycenter FM score",
    title="ν selection optimization",
    signal_name=r"$\nu$",
    bkg_name="Cosmic background",
    xlim=(0.002, 0.2),
    nbins=25,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False,
)

fig.savefig(file_dir + "/bfm_optimization.png", dpi=300)
plt.show()

# Nu Score

In [ ]:
# Drop the last index level (rec.slc.reco.pfp..index)
mask = (evt_df.slc.cut.obvious_cosmic == True) &  (evt_df.slc.cut.inside_FV == True) 
evt_df_reset = evt_df[mask].reset_index(level='rec.slc.reco.pfp..index', drop=True)

# Drop duplicates based on the remaining index levels
evt_df_unique = evt_df_reset[~evt_df_reset.index.duplicated(keep='first')]

# Split signal and background
#signal_df = evt_df_unique[(evt_df_unique.truth.nu_categ == "CC1pi")]
#bkg_df    = evt_df_unique[(evt_df_unique.truth.nu_categ != "CC1pi")]
signal_df = evt_df_unique[(evt_df_unique.truth.nu_categ != "cosmic")]
bkg_df    = evt_df_unique[(evt_df_unique.truth.nu_categ == "cosmic")]

column = ('slc', 'nu_score', '', '', '', '')

df_opt, best,fig = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column=column,
    cut_type=">",
    xlabel=r"$\nu$ Score",
    title="ν selection optimization",
    signal_name=r"$\nu$",
    bkg_name="cosmic",
    xlim=(0, 1),
    nbins=50,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False,
)

fig.savefig(file_dir + "/nu_score_optimization_neutrinos.png", dpi=300)
plt.show()

In [ ]:
# Drop the last index level (rec.slc.reco.pfp..index)

mask_until_t0 = (evt_df.slc.cut.inside_FV == True)& (evt_df.slc.cut.obvious_cosmic ==  True) &(evt_df.slc.cut.t0 == True)

mask_full_sel = (
   (evt_df.slc.cut.inside_FV == True)& (evt_df.slc.cut.obvious_cosmic ==  True) &(evt_df.slc.cut.t0 == True) 
    & (evt_df.slc.cut.track == True)  & (evt_df.slc.cut.MIP_candidates == True) & (evt_df.slc.cut.shower == True)
    &  (evt_df.slc.cut.containment == True) 
    & (evt_df.slc.cut.angle == True) 
    & (evt_df.slc.cut.proton_BDT == True)
    & (evt_df.slc.cut.michel == True) 
    & (evt_df.slc.cut.extra_pion == True)    
)

mask_not_angle_applied = (
    (evt_df.slc.cut.inside_FV == True)& (evt_df.slc.cut.obvious_cosmic ==  True) &(evt_df.slc.cut.t0 == True) 
    & (evt_df.slc.cut.track == True)  & (evt_df.slc.cut.MIP_candidates == True) & (evt_df.slc.cut.shower == True)
    & (evt_df.slc.cut.containment == True) 
    & (evt_df.slc.cut.proton_BDT == True)
    & (evt_df.slc.cut.michel == True) 
    & (evt_df.slc.cut.extra_pion == True)    
)

mask_up_to_MIP_id = (
    (evt_df.slc.cut.inside_FV == True)& (evt_df.slc.cut.obvious_cosmic ==  True) &(evt_df.slc.cut.t0 == True) 
    & (evt_df.slc.cut.track == True)  & (evt_df.slc.cut.MIP_candidates == True) & (evt_df.slc.cut.shower == True)
    & (evt_df.slc.cut.containment == True) 
)

cut_mask_and_names = {
    'until_t0': mask_until_t0,
    'mask_not_angle_applied': mask_not_angle_applied,
    'full_selection': mask_full_sel,
    'up_to_MIP_id': mask_up_to_MIP_id
}

for name, mask in cut_mask_and_names.items():
    evt_df_reset = evt_df[mask].reset_index(level='rec.slc.reco.pfp..index', drop=True)
    
    # Drop duplicates based on the remaining index levels
    evt_df_unique = evt_df_reset[~evt_df_reset.index.duplicated(keep='first')]
    
    # Split signal and background
    signal_df = evt_df_unique[ ((evt_df_unique.truth.nu_categ == "CC1pi") | (evt_df_unique.truth.nu_categ == "other_CC1pi"))]
    bkg_df    = evt_df_unique[ ((evt_df_unique.truth.nu_categ != "CC1pi") & (evt_df_unique.truth.nu_categ != "other_CC1pi"))]
    
    #signal_df = evt_df_unique[(evt_df_unique.truth.nu_categ != "cosmic")]
    #bkg_df    = evt_df_unique[(evt_df_unique.truth.nu_categ == "cosmic")]
    
    column = ('slc', 'nu_score', '', '', '', '')
    
    df_opt, best,fig = OptimizationUtils.optimize_cut_eff_pur(
        signal_df,
        bkg_df,
        column=column,
        cut_type=">",
        xlabel=r"$\nu$ score",
        title="ν selection optimization",
        signal_name=r"$\nu_{\mu}CC1\pi$",
        bkg_name="background",
        xlim=(0, 1),
        nbins=50,
        legend_loc="upper right",
        cut_unit="",
        normalize_hist = False,
    )
    
    
    fig.savefig(file_dir + "/nu_score_optimization_cc1pi" + name + ".png", dpi=300)
    plt.show()

# Chi2 cut

In [ ]:
SLICE_LEVELS = ["__ntuple", "entry", "rec.slc..index"]
chi2_dir = "/exp/sbnd/data/users/lpelegri/Graphs/Optimization/Chi2"
os.makedirs(chi2_dir, exist_ok=True)  # create directory if needed

In [ ]:
# -------------------------
# Select signal and background
# -------------------------
cut_mask = (evt_df.slc.cut.nu_score == True) & (evt_df.slc.cut.t0 == True) & (evt_df.slc.cut.track == True) & (evt_df.slc.cut.inside_FV == True) & (evt_df.slc.cut.obvious_cosmic == True) 

signal_df = evt_df[ cut_mask&
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

bkg_df = evt_df[ cut_mask&
    ((abs(evt_df.pfp.trk.truth.p.pdg) != 211) & (abs(evt_df.pfp.trk.truth.p.pdg) != 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]



# -------------------------
# Columns
# -------------------------
col_chi2_mu = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p  = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_len  = ('pfp', 'trk', 'len', '', '', '')


# -------------------------
# Remove rows with NaN or <=0 in the columns of interest
# -------------------------
signal_df = signal_df[(signal_df[col_chi2_mu] > 0) & (signal_df[col_chi2_p] > 0)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] > 0) & (bkg_df[col_chi2_p] > 0)]

# -------------------------
# Manual 2D cut scan
# -------------------------
best_2d, results_df = OptimizationUtils.manual_cut_optimizer(
    signal_df=signal_df,
    bkg_df=bkg_df,
    cut_columns=[col_chi2_mu, col_chi2_p],
    cut_sign=["<", ">"],          # "<" for chi2_mu, ">" for chi2_proton
    cut_min=[0, 0],               # min scan range
    cut_max=[60, 300],            # max scan range
    n_steps= [61,101],
    min_eff=0.0,
    min_pur=0.0,  # <-- new parameter                 # number of steps in each variable
)


# -------------------------
# Manual 2D cut scan
# -------------------------
best3d, results_df3d = OptimizationUtils.manual_cut_optimizer(
    signal_df=signal_df,
    bkg_df=bkg_df,
    cut_columns=[col_chi2_mu, col_chi2_p,col_len],
    cut_sign=["<", ">", ">"],          # "<" for chi2_mu, ">" for chi2_proton
    cut_min=[30, 70,3],               # min scan range
    cut_max=[50, 120,20],            # max scan range
    n_steps= [21,51,18],
    min_eff=0.0,
    min_pur=0.0,  # <-- new parameter                 # number of steps in each variable
)


In [ ]:
# Split signal and background

cut_mask = (evt_df.slc.cut.nu_score == True) & (evt_df.slc.cut.t0 == True) & (evt_df.slc.cut.track == True) & (evt_df.slc.cut.inside_FV == True) & (evt_df.slc.cut.obvious_cosmic == True)

signal_df = evt_df[cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

bkg_df = evt_df[cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) != 211) & (abs(evt_df.pfp.trk.truth.p.pdg) != 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

# -------------------------
# Columns
# -------------------------
col_chi2_mu = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p  = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_len  = ('pfp', 'trk', 'len', '', '', '')


# -------------------------
# Remove rows with NaN or <=0 in the columns of interest
# -------------------------
signal_df = signal_df[(signal_df[col_chi2_mu] > 0) & (signal_df[col_chi2_p] > 0)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] > 0) & (bkg_df[col_chi2_p] > 0)]
signal_df = signal_df[(signal_df[col_chi2_mu] < best_2d["cuts"][0]) & (signal_df[col_chi2_p] > best_2d["cuts"][1])]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] < best_2d["cuts"][0]) & (bkg_df[col_chi2_p] > best_2d["cuts"][1])]

df_opt_len, best_len,fig = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column= col_len,
    cut_type=">",
    xlabel="length",
    title="pfp optimization",
    signal_name=r"$\mu / \pi$",
    bkg_name="Other",
    xlim=(0, 30),
    nbins=31,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False,
    min_pur=0.0 # <-- new parameter
)
fig.savefig(chi2_dir + "/lenop_plus_2d.png", dpi=300)
plt.show()

In [ ]:

cut_mask = (evt_df.slc.cut.nu_score == True) & (evt_df.slc.cut.t0 == True) & (evt_df.slc.cut.track == True) & (evt_df.slc.cut.inside_FV == True) & (evt_df.slc.cut.obvious_cosmic == True) 

signal_df = evt_df[cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

bkg_df = evt_df[cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) != 211) & (abs(evt_df.pfp.trk.truth.p.pdg) != 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]
OptimizationUtils.plot_2d_cut_individual_manual(
    signal_df,
    bkg_df,
    col1=col_chi2_mu,
    col2=col_chi2_p,
    cutnum1=0,
    cutnum2=1,
    best=best_2d,
    xlabel=r"$\chi^2_\mu$",
    ylabel=r"$\chi^2_p$",
    signal_label="Signal",
    bkg_label="Background",
    xlim=(0, 60),
    ylim=(0, 300),
    bins=60,
    cmap= sunset_cmap
)

OptimizationUtils.plot_2d_cut_metric_heatmap(
    signal_df,
    bkg_df,
    cutnum1=0,
    cutnum2=1,
    best=best_2d,
    results_df=results_df,
    xlabel=r"$\chi^2_\mu$",
    ylabel=r"$\chi^2_p$",
    signal_label="Signal",
    bkg_label="Background",
    xlim=(0, 60),
    ylim=(0, 300),
    bins=60,
    cmap= sunset_cmap,
    save_dir = chi2_dir
)

In [ ]:
# Split signal and background

cut_mask = (evt_df.slc.cut.nu_score == True) & (evt_df.slc.cut.t0 == True) & (evt_df.slc.cut.track == True)& (evt_df.slc.cut.inside_FV == True) & (evt_df.slc.cut.obvious_cosmic == True)

signal_df = evt_df[cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

bkg_df = evt_df[cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) != 211) & (abs(evt_df.pfp.trk.truth.p.pdg) != 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

# -------------------------
# Columns
# -------------------------
col_chi2_mu = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p  = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_len  = ('pfp', 'trk', 'len', '', '', '')


# -------------------------
# Remove rows with NaN or <=0 in the columns of interest
# -------------------------
signal_df = signal_df[(signal_df[col_chi2_mu] > 0) & (signal_df[col_chi2_p] > 0)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] > 0) & (bkg_df[col_chi2_p] > 0)]
signal_df = signal_df[(signal_df[col_chi2_mu] < 20) & (signal_df[col_len] > 10)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] < 20) & (bkg_df[col_len] > 10)]

df_opt_len, best_chi2,fig = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column= col_chi2_p,
    cut_type=">",
    xlabel=r"$\chi^2_p$",
    title="pfp optimization",
    signal_name=r"$\mu / \pi$",
    bkg_name="Other",
    xlim=(0, 300),
    nbins=51,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False,
    min_pur=0.0 # <-- new parameter
)
fig.savefig(chi2_dir + "/fixedlen_fixedchi2mu_chi2pop.png", dpi=300)
plt.show()

# Plot the confusion matrix

In [ ]:
development_sample_file = "/exp/sbnd/data/users/lpelegri/cafpyana_data/wrong_energy_cut/mc_ar23p_pruned_no_syst_for_BDT_comp.df"

development_sample_df = load_df(development_sample_file, keys2load, 100, filter_df = False, reprocess_df = False, reprocess_truth = False)
dev_sample_evt_df = development_sample_df['cc1pi']
dev_sample_hdr_df = development_sample_df['hdr']
#dev_sample_nu_df = development_sample_df['nudf']

print("data_tot_pot: %.3e" %(data_tot_pot))

dev_sample_tot_pot = dev_sample_hdr_df['pot'].sum()
mc_pot_scale = data_tot_pot / dev_sample_tot_pot
print("mc_tot_pot: %.3e" %(dev_sample_tot_pot))
print("mc_pot_scale: %.3e" %(mc_pot_scale))
dev_sample_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(dev_sample_evt_df))

In [ ]:
cut_mask = (dev_sample_evt_df.slc.cut.nu_score == True) & (dev_sample_evt_df.slc.cut.t0 == True) & (dev_sample_evt_df.slc.cut.track == True)& (dev_sample_evt_df.slc.cut.inside_FV == True) & (dev_sample_evt_df.slc.cut.obvious_cosmic == True)
signal_df = dev_sample_evt_df[cut_mask &
    ((abs(dev_sample_evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(dev_sample_evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(dev_sample_evt_df.pfp.trk.len) > 3) & (abs(dev_sample_evt_df.pfp.trackScore) > 0.5) & (dev_sample_evt_df.pfp.dist_to_vertex < 10)
]

bkg_df = dev_sample_evt_df[cut_mask &
    ((abs(dev_sample_evt_df.pfp.trk.truth.p.pdg) != 211) & (abs(dev_sample_evt_df.pfp.trk.truth.p.pdg) != 13)) &
    (abs(dev_sample_evt_df.pfp.trk.len) > 3) & (abs(dev_sample_evt_df.pfp.trackScore) > 0.5) & (dev_sample_evt_df.pfp.dist_to_vertex < 10)
]

print("Fixed 85")
cm = ConfusionMatricesUtils.build_confusion_matrix_from_cuts(
    signal_df=signal_df,
    bkg_df=bkg_df,
    cols=[col_chi2_mu, col_chi2_p,col_len],
    cuts=[20,85,10],
    cut_sign=["<", ">", ">"],          # "<" for chi2_mu, ">" for chi2_proton
)
fig = ConfusionMatricesUtils.plot_confusion_matrix(
    cm = cm,
    x_labels = [r"$\mu/\pi$","other"],
    y_labels = [r"$\mu/\pi$","other"]
)
fig.savefig(chi2_dir + "/confusion_matrix_def.png", dpi=300)
plt.show()

best_fixed = {"cuts": [20,85]}
OptimizationUtils.plot_2d_cut_individual_manual(
    signal_df,
    bkg_df,
    col1=col_chi2_mu,
    col2=col_chi2_p,
    cutnum1=0,
    cutnum2=1,
    save_dir= chi2_dir + "/chi2mu_chi2p_",
    best=best_fixed,
    xlabel=r"$\chi^2_\mu$",
    ylabel=r"$\chi^2_p$",
    signal_label=r"$\mu/\pi$",
    bkg_label="other",
    xlim=(0, 60),
    ylim=(0, 300),
    bins=60,
    cmap= sunset_cmap
)

print("3D OPT")
cm = ConfusionMatricesUtils.build_confusion_matrix_from_cuts(
    signal_df=signal_df,
    bkg_df=bkg_df,
    cols=[col_chi2_mu, col_chi2_p,col_len],
    cuts=best3d["cuts"],
    cut_sign=["<", ">", ">"],          # "<" for chi2_mu, ">" for chi2_proton
)
fig =  ConfusionMatricesUtils.plot_confusion_matrix(
    cm = cm,
    x_labels = [r"$\mu/\pi$","other"],
    y_labels = [r"$\mu/\pi$","other"]
)
fig.savefig(chi2_dir + "/confusion_matrix_3dop.png", dpi=300)
plt.show()


print("2D OPT + Len")
print(np.append(best_2d["cuts"], best_len["cut"]))
cm = ConfusionMatricesUtils.build_confusion_matrix_from_cuts(
    signal_df=signal_df,
    bkg_df=bkg_df,
    cols=[col_chi2_mu, col_chi2_p,col_len],
    cuts = np.append(best_2d["cuts"], best_len["cut"]),
    cut_sign=["<", ">", ">"],          # "<" for chi2_mu, ">" for chi2_proton
)
fig =  ConfusionMatricesUtils.plot_confusion_matrix(
    cm = cm,
    x_labels = [r"$\mu/\pi$","other"],
    y_labels = [r"$\mu/\pi$","other"]
)
fig.savefig(chi2_dir + "/confusion_matrix_2dop_len.png", dpi=300)
plt.show()


In [ ]:
def plot_2d_by_category(
    df,
    col1,
    col2,
    category_col,
    best,
    cutnum1,
    cutnum2,
    save_dir=None,
    xlabel="Variable 1",
    ylabel="Variable 2",
    xlim=None,
    ylim=None,
    bins=50,
    cmap="plasma"
):
    """
    Groups a DataFrame by 'category_col', produces a 2D histogram 
    for every unique category, including a 'MIP region' label,
    and saves them individually.
    """
    import os
    import matplotlib.pyplot as plt
    import numpy as np

    # 1. Setup Directory
    if save_dir and not os.path.exists(save_dir):
        os.makedirs(save_dir)

    # 2. Extract Cut values from 'best' dict
    cut1_best = best["cuts"][cutnum1]
    cut2_best = best["cuts"][cutnum2]
    hist_range = [xlim, ylim] if xlim and ylim else None

    # 3. Get unique categories, sorted
    categories = sorted(df[category_col].unique())
    print(f"Found categories: {categories}")

    # 4. Iterate and Plot
    for cat in categories:
        # Filter data for this specific category
        cat_df = df[df[category_col] == cat].loc[:, [col1, col2]].dropna()
        
        if cat_df.empty:
            print(f"Skipping {cat}: No data after dropna.")
            continue

        fig, ax = plt.subplots(figsize=(7, 6))
        
        # Create the 2D Histogram
        im = ax.hist2d(
            cat_df[col1], 
            cat_df[col2], 
            bins=bins, 
            range=hist_range, 
            cmap=cmap
        )
        
        # Add colorbar
        plt.colorbar(im[3], ax=ax, label="Events")
        
        # --- NEW: MIP REGION LABEL ---
        # (0.02, 0.95) places it at 2% from the left, 95% from the bottom of the axes.
        '''
        I’m ax.text(
            0.02, 0.98, 
            "MIP region", 
            transform=ax.transAxes, 
            fontsize=12, 
            fontweight='bold', 
            color='black', # Changed to white for better contrast with plasma cmap
            va='top', 
            bbox=dict(facecolor='white', edgecolor='black') # Added subtle background box
        )
        '''
        
        # Plot cut lines
        ax.axvline(cut1_best, color="black", linestyle="--", linewidth=2.0)
        ax.axhline(cut2_best, color="black", linestyle="--", linewidth=2.0)
        
        # Labels and Title
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(f"{cat}")
        plt.tight_layout()

        # 5. Save Logic
        if save_dir:
            # Clean category name for filename
            clean_name = str(cat).replace(" ", "_").replace("/", "_")
            filename = f"hist2d_MIP_{clean_name}.pdf"
            path = os.path.join(save_dir, filename)
            plt.savefig(path, format='pdf', dpi=300)
            print(f"Saved: {path}")
        
        plt.show()

    # --- Total Plot (Combined) ---
    fig_tot, ax_tot = plt.subplots(figsize=(7, 6))
    total_df = df.loc[:, [col1, col2]].dropna()
    im_tot = ax_tot.hist2d(total_df[col1], total_df[col2], bins=bins, range=hist_range, cmap=cmap)
    plt.colorbar(im_tot[3], ax=ax_tot, label="Total Events")
    
    # Add MIP region label to the total plot too
    '''
    ax_tot.text(0.02, 0.98, "MIP region", transform=ax_tot.transAxes, 
                fontsize=12, fontweight='bold', color='black', va='top', 
                bbox=dict(facecolor='white', alpha=0.3, edgecolor='black'))
    '''
    
    ax_tot.axvline(cut1_best, color="black", linestyle="--", linewidth=2.0)
    ax_tot.axhline(cut2_best, color="black", linestyle="--", linewidth=2.0)
    ax_tot.set_xlabel(xlabel)
    ax_tot.set_ylabel(ylabel)
    ax_tot.set_title("Total (All Categories Combined)")
    plt.tight_layout()
    
    if save_dir:
        plt.savefig(os.path.join(save_dir, "hist2d_total_combined.pdf"), format='pdf', dpi=300)
    
    plt.show()

In [ ]:
cut_mask = (evt_df.slc.cut.nu_score == True) & (evt_df.slc.cut.t0 == True) & (evt_df.slc.cut.track == True) & (evt_df.slc.cut.inside_FV == True) & (evt_df.slc.cut.obvious_cosmic == True)
df = evt_df[cut_mask &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
    & (evt_df.pfp.is_exiting == False)
]

pdg_col = ('pfp', 'trk', 'truth', 'p', 'pdg', '')
endp_col = ('pfp', 'trk', 'truth', 'p', 'end_process', '')
p_type_col =  ('pfp', 'trk', 'truth', 'p', 'p_type', '')

print(df.loc[(abs(df.pfp.trk.truth.p.pdg) == 2212), endp_col])
df.loc[(abs(df.pfp.trk.truth.p.pdg) == 2212) & (df.pfp.trk.truth.p.end_process == 7),p_type_col] = "inelastic proton"
df.loc[(abs(df.pfp.trk.truth.p.pdg) == 2212) & (df.pfp.trk.truth.p.end_process == 45),p_type_col] = "stopping proton"

dir_p = "/exp/sbnd/data/users/lpelegri/Graphs/ProtonBDT"
os.makedirs(dir_p, exist_ok=True)  # create directory if needed
plot_2d_by_category(
    df,
    col1=col_chi2_mu,
    col2=col_chi2_p,
    category_col = p_type_col,
    cutnum1=0,
    cutnum2=1,
    save_dir= dir_p + "/chi2mu_chi2p",
    best=best_fixed,
    xlabel=r"$\chi^2_\mu$",
    ylabel=r"$\chi^2_p$",
    xlim=(0, 60),
    ylim=(0, 300),
    bins=60,
    cmap= sunset_cmap
)



# Shower cut

In [ ]:
def shower_cut_mask(df, group_levels, min_shower_ke = 0):
    is_pandora_primary_mask = (df.pfp.parent_is_primary == True)
    is_shower_mask = (df.pfp.trackScore >= 0) & (df.pfp.trackScore < CTE.max_shower_track_score)
    energy_mask = (df.pfp.shw.bestplane_energy > min_shower_ke)
    shower_df = df[is_pandora_primary_mask & is_shower_mask & energy_mask] 
        
    # Count how many pfps per slice
    shower_counts = shower_df.groupby(level=group_levels).size()
    
    # Get only slices with at least 2 pfps
    non_valid_slices = shower_counts[(shower_counts > 0)].index

    # Apply the mask to original DataFrame
    final_mask = pd.Series(~df.index.droplevel('rec.slc.reco.pfp..index').isin(non_valid_slices), index=df.index)
   
    return final_mask

In [ ]:
# Drop the last index level (rec.slc.reco.pfp..index)
# Split signal and background
#cut_mask = (evt_df.slc.nu_score > 0.55) & (evt_df.slc.barycenterFM.score > 0.03)

cut_mask = (evt_df.slc.cut.nu_score == True) & (evt_df.slc.cut.t0 == True) & (evt_df.slc.cut.inside_FV == True) & (evt_df.slc.cut.obvious_cosmic == True)  #& (evt_df.slc.cut.track == True)  & (evt_df.slc.cut.MIP_candidates == True)

#cut_mask = True
signal_df = evt_df[cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 11) & (abs(evt_df.pfp.trackScore) < 0.5) & (evt_df.pfp.parent_is_primary == True))
]

bkg_df = evt_df[cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 22) & (abs(evt_df.pfp.trackScore) < 0.5) & (evt_df.pfp.parent_is_primary == True))
]

df_opt, best, fig = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column=('pfp', 'shw', 'bestplane_energy', '', '', ''),
    cut_type="<",
    xlabel="best plane energy [GeV]",
    title="ν selection optimization",
    signal_name=r"e",
    bkg_name=r"$\gamma$",
    xlim=(0, 0.2),
    nbins=25,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False,
)


fig.savefig(file_dir + "/shower_op.png", dpi=300)


In [ ]:

cut_mask = (evt_df.slc.cut.nu_score == True) & (evt_df.slc.cut.t0 == True)& (evt_df.slc.cut.inside_FV == True) & (evt_df.slc.cut.obvious_cosmic == True)  & (evt_df.slc.cut.track == True)  & (evt_df.slc.cut.MIP_candidates == True)

print("Control")
HelperFunctions.print_category_metrics(evt_df, evt_df[cut_mask], target_categ="CC1pi")
print(0.075)
HelperFunctions.print_category_metrics(evt_df, evt_df[cut_mask & shower_cut_mask(evt_df, SLICE_LEVELS, min_shower_ke = 0.075)], target_categ="CC1pi")
print(0.07)
HelperFunctions.print_category_metrics(evt_df, evt_df[cut_mask & shower_cut_mask(evt_df, SLICE_LEVELS, min_shower_ke = 0.070)], target_categ="CC1pi")
print(0.065)
HelperFunctions.print_category_metrics(evt_df, evt_df[cut_mask & shower_cut_mask(evt_df, SLICE_LEVELS, min_shower_ke = 0.065)], target_categ="CC1pi")
print(0.06)
HelperFunctions.print_category_metrics(evt_df, evt_df[cut_mask & shower_cut_mask(evt_df, SLICE_LEVELS, min_shower_ke = 0.06)], target_categ="CC1pi")
print(0.055)
HelperFunctions.print_category_metrics(evt_df, evt_df[cut_mask & shower_cut_mask(evt_df, SLICE_LEVELS, min_shower_ke = 0.055)], target_categ="CC1pi")
print(0.045)
HelperFunctions.print_category_metrics(evt_df, evt_df[cut_mask & shower_cut_mask(evt_df, SLICE_LEVELS, min_shower_ke = 0.045)], target_categ="CC1pi")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

cut_mask = (evt_df.slc.cut.nu_score == True) & (evt_df.slc.cut.t0 == True)& (evt_df.slc.cut.inside_FV == True) & (evt_df.slc.cut.obvious_cosmic == True)  & (evt_df.slc.cut.track == True)  & (evt_df.slc.cut.MIP_candidates == True)

# 1. Define range of cut values from 0.1 to 0.5 (adjust step as needed)
cut_values = np.linspace(0.03, 0.2, 35) 

purities = []
efficiencies = []
pur_x_eff = []

# 2. Loop through cut values and collect metrics
for cut in cut_values:
    print(cut)
    mask = cut_mask & shower_cut_mask(evt_df, SLICE_LEVELS, min_shower_ke=cut)
    metrics = HelperFunctions.print_category_metrics(evt_df, evt_df[mask], target_categ="CC1pi")
    
    # Unpack metrics (assuming metrics dict contains 'purity' and 'efficiency')
    pur = metrics["pur"]
    eff = metrics["eff"]
    pxe = pur * eff
    
    purities.append(pur)
    efficiencies.append(eff)
    pur_x_eff.append(pxe)

# Convert to arrays for easy indexing
purities = np.array(purities)
efficiencies = np.array(efficiencies)
pur_x_eff = np.array(pur_x_eff)

# 3. Find the maximum Purity x Efficiency
max_idx = np.argmax(pur_x_eff)
best_cut = cut_values[max_idx]
best_pxe = pur_x_eff[max_idx]

# 4. Plotting
plt.figure(figsize=(9, 5), dpi=100)

plt.plot(cut_values, purities, label="Purity", color="#1f77b4", linewidth=2)
plt.plot(cut_values, efficiencies, label="Efficiency", color="#2ca02c", linewidth=2)
plt.plot(cut_values, pur_x_eff, label="Purity $\\times$ Efficiency", color="#ff7f0e", linewidth=2.5)

# Highlight Maximum Purity x Efficiency
plt.axvline(x=best_cut, color="red", linestyle="--", alpha=0.7, label=f"Max Cut = {best_cut:.3f}")
plt.plot(best_cut, best_pxe, marker="o", markersize=8, color="red", label=f"Max P$\\times$E = {best_pxe:.3f}")

# Styling
plt.xlabel("Min Shower KE Cut Value", fontsize=12)
plt.ylabel("Metric Score", fontsize=12)
plt.title("Cut Optimization: Purity vs Efficiency", fontsize=14, pad=10)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="best", frameon=True)
plt.tight_layout()

plt.show()

# Angle Cut 

In [ ]:
# Drop the last index level (rec.slc.reco.pfp..index)

mask_until_t0 = (evt_df.slc.cut.inside_FV == True)& (evt_df.slc.cut.obvious_cosmic ==  True) &(evt_df.slc.cut.t0 == True)

mask_full_sel = (
   (evt_df.slc.cut.inside_FV == True)& (evt_df.slc.cut.obvious_cosmic ==  True) &(evt_df.slc.cut.t0 == True) 
    & (evt_df.slc.cut.track == True)  & (evt_df.slc.cut.MIP_candidates == True) & (evt_df.slc.cut.shower == True)
    &  (evt_df.slc.cut.containment == True) 
    & (evt_df.slc.cut.nu_score == True) 
    & (evt_df.slc.cut.proton_BDT == True)
    & (evt_df.slc.cut.michel == True) 
    & (evt_df.slc.cut.extra_pion == True)    
)

mask_not_angle_applied = (
    (evt_df.slc.cut.inside_FV == True)& (evt_df.slc.cut.obvious_cosmic ==  True) &(evt_df.slc.cut.t0 == True) 
    & (evt_df.slc.cut.track == True)  & (evt_df.slc.cut.MIP_candidates == True) & (evt_df.slc.cut.shower == True)
    & (evt_df.slc.cut.containment == True) 
    & (evt_df.slc.cut.proton_BDT == True)
    & (evt_df.slc.cut.michel == True) 
    & (evt_df.slc.cut.extra_pion == True)    
)

mask_no_containment = (
    (evt_df.slc.cut.inside_FV == True)& (evt_df.slc.cut.obvious_cosmic ==  True) &(evt_df.slc.cut.t0 == True) 
    & (evt_df.slc.cut.track == True)  & (evt_df.slc.cut.MIP_candidates == True) & (evt_df.slc.cut.shower == True)
)

cut_mask_and_names = {
    'until_t0': mask_until_t0,
    'mask_not_nu_score_applied': mask_not_angle_applied,
    'full_selection': mask_full_sel,
    'mask_no_containment':mask_no_containment
}

for name, mask in cut_mask_and_names.items():
    
    
    # Drop the last index level (rec.slc.reco.pfp..index)
    evt_df_reset = evt_df[mask].reset_index(level='rec.slc.reco.pfp..index', drop=True)
    
    # Drop duplicates based on the remaining index levels
    evt_df_unique = evt_df_reset[~evt_df_reset.index.duplicated(keep='first')]
    
    
    # Split signal and background
    signal_df = evt_df_unique[
        (evt_df_unique.truth.nu_categ == "CC1pi") |
        (evt_df_unique.truth.nu_categ == "other_CC1pi")
    ]
    bkg_df    = evt_df_unique[((evt_df_unique.truth.nu_categ != "CC1pi") & (evt_df_unique.truth.nu_categ != "other_CC1pi"))]
    #signal_df = evt_df_unique[(evt_df_unique.truth.nu_categ != "cosmic")]
    #bkg_df    = evt_df_unique[(evt_df_unique.truth.nu_categ == "cosmic")]
    
    column = ('slc', 'measure_var', 'angle_between_candidates', '', '', '')
    
    df_opt, best, fig = OptimizationUtils.optimize_cut_eff_pur(
        signal_df,
        bkg_df,
        column=column,
        cut_type="<",
        xlabel="Angle between candidates [rad]",
        title="ν selection optimization",
        signal_name=r"$\nu_{\mu}CC1\pi$",
        bkg_name="background",
        xlim=(0, 3.2),
        nbins=40,
        legend_loc="upper right",
        cut_unit="[rad]",
        normalize_hist = False,
    )
    fig.savefig(file_dir + "/angle_optimization_cc1pi" + name + ".png", dpi=300)
    plt.show()

# Michel removal selection

In [ ]:
cut_mask = (
    (evt_df.slc.cut.inside_FV == True)& (evt_df.slc.cut.obvious_cosmic ==  True) &(evt_df.slc.cut.t0 == True) 
    & (evt_df.slc.cut.nu_score == True) 
    & (evt_df.slc.cut.track == True)  & (evt_df.slc.cut.MIP_candidates == True) & (evt_df.slc.cut.shower == True)
    & (evt_df.slc.cut.proton_BDT == True)
    &  (evt_df.slc.cut.containment == True) 
    #& (evt_df.slc.cut.angle == True)
)


bkg_df = evt_df[
    cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

signal_df = evt_df[
    cut_mask & 
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 11) | (abs(evt_df.pfp.trk.truth.p.pdg) == 22)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

# -------------------------
# Columns
# -------------------------
col_chi2_mu = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p  = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_len  = ('pfp', 'trk', 'len', '', '', '')

col_num_hits  = ('pfp', 'max_daughter_hits', '', '', '', '')
col_mean_dEdx  = ('pfp', 'trk', 'mean_dEdx', '', '', '')
col_best_ke  = ('pfp', 'trk', 'calo', 'best', 'ke', '')
col_trk_score  = ('pfp', 'trackScore', '', '', '', '')


# -------------------------
# Remove rows with NaN or <=0 in the columns of interest
# -------------------------
signal_df = signal_df[(signal_df[col_chi2_mu] > 0) & (signal_df[col_chi2_p] > 0)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] > 0) & (bkg_df[col_chi2_p] > 0)]
signal_df = signal_df[(signal_df[col_chi2_mu] < 20) & (signal_df[col_chi2_p] > 85)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] < 20) & (bkg_df[col_chi2_p] > 85)]
signal_df = signal_df[(signal_df[col_num_hits] == 0)]
bkg_df    = bkg_df[bkg_df[col_num_hits] ==0]
signal_df = signal_df[(signal_df[col_len] > 10)]
bkg_df    = bkg_df[bkg_df[col_len] > 10]

best_3D, results_df = OptimizationUtils.manual_cut_optimizer(
    signal_df=signal_df,
    bkg_df=bkg_df,
    cut_columns=[col_mean_dEdx, col_best_ke,col_trk_score],
    cut_sign=["<", "<" , "<"],          # "<" for chi2_mu, ">" for chi2_proton
    cut_min=[0, 0, 0.5],               # min scan range
    cut_max=[10, 100, 0.7],            # max scan range
    n_steps= [11,21, 5],
    min_eff=0.0,
    min_pur=0.0,  # <-- new parameter                 # number of steps in each variable
)

best_2D, results_df = OptimizationUtils.manual_cut_optimizer(
    signal_df=signal_df,
    bkg_df=bkg_df,
    cut_columns=[col_best_ke,col_trk_score],
    cut_sign=[ "<" , "<"],          # "<" for chi2_mu, ">" for chi2_proton
    cut_min=[ 0, 0.5],               # min scan range
    cut_max=[ 100, 0.7],            # max scan range
    n_steps= [101, 21],
    min_eff=0.0,
    min_pur=0.0,  # <-- new parameter                 # number of steps in each variable
)


In [ ]:
OptimizationUtils.plot_2d_cut_metric_heatmap(
    signal_df,
    bkg_df,
    cutnum1=0,
    cutnum2=1,
    best=best_2D,
    results_df=results_df,
    xlabel=r"KE [MeV]",
    ylabel=r"Track score",
    signal_label="Signal",
    bkg_label="Background",
    xlim=(0, 60),
    ylim=(0, 300),
    bins=60,
    cmap= sunset_cmap,
    save_dir = file_dir
)

# Confusion matrices

In [ ]:
cut_mask = (
    (dev_sample_evt_df.slc.cut.inside_FV == True)& (dev_sample_evt_df.slc.cut.obvious_cosmic ==  True) &(dev_sample_evt_df.slc.cut.t0 == True) 
    & (dev_sample_evt_df.slc.cut.nu_score == True) 
    & (dev_sample_evt_df.slc.cut.track == True)  & (dev_sample_evt_df.slc.cut.MIP_candidates == True) & (dev_sample_evt_df.slc.cut.shower == True)
    & (dev_sample_evt_df.slc.cut.proton_BDT == True)
    & (dev_sample_evt_df.slc.cut.containment == True) 
    & (dev_sample_evt_df.slc.cut.angle == True)
)


bkg_df = dev_sample_evt_df[
    cut_mask &
    ((abs(dev_sample_evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(dev_sample_evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(dev_sample_evt_df.pfp.trk.len) > 3) & (abs(dev_sample_evt_df.pfp.trackScore) > 0.5) & (dev_sample_evt_df.pfp.dist_to_vertex < 10)
]

signal_df = dev_sample_evt_df[
    cut_mask & 
    ((abs(dev_sample_evt_df.pfp.trk.truth.p.pdg) == 11) | (abs(dev_sample_evt_df.pfp.trk.truth.p.pdg) == 22)) &
    (abs(dev_sample_evt_df.pfp.trk.len) > 3) & (abs(dev_sample_evt_df.pfp.trackScore) > 0.5) & (dev_sample_evt_df.pfp.dist_to_vertex < 10)
]

# -------------------------
# Columns
# -------------------------
col_chi2_mu = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p  = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_len  = ('pfp', 'trk', 'len', '', '', '')

col_num_hits  = ('pfp', 'max_daughter_hits', '', '', '', '')
col_mean_dEdx  = ('pfp', 'trk', 'mean_dEdx', '', '', '')
col_best_ke  = ('pfp', 'trk', 'calo', 'best', 'ke', '')
col_trk_score  = ('pfp', 'trackScore', '', '', '', '')


# -------------------------
# Remove rows with NaN or <=0 in the columns of interest
# -------------------------
signal_df = signal_df[(signal_df[col_chi2_mu] > 0) & (signal_df[col_chi2_p] > 0)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] > 0) & (bkg_df[col_chi2_p] > 0)]
signal_df = signal_df[(signal_df[col_chi2_mu] < 20) & (signal_df[col_chi2_p] > 85)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] < 20) & (bkg_df[col_chi2_p] > 85)]
signal_df = signal_df[(signal_df[col_num_hits] == 0)]
bkg_df    = bkg_df[bkg_df[col_num_hits] ==0]
signal_df = signal_df[(signal_df[col_len] > 10)]
bkg_df    = bkg_df[bkg_df[col_len] > 10]



print("OG")
cm = ConfusionMatricesUtils.build_confusion_matrix_from_cuts(
    signal_df=signal_df,
    bkg_df=bkg_df,
    cols=[col_mean_dEdx, col_best_ke,col_trk_score],
    cuts=[2.75,50,0.60],
    cut_sign=["<", "<" , "<"], 
)

ConfusionMatricesUtils.plot_confusion_matrix(
    cm = cm,
    x_labels = ["e/photon",r"$\mu/\pi$"],
    y_labels = ["e/photon",r"$\mu/\pi$"],
    cmap=sunset_cmap
)


print("3D")
cm = ConfusionMatricesUtils.build_confusion_matrix_from_cuts(
    signal_df=signal_df,
    bkg_df=bkg_df,
    cols=[col_mean_dEdx, col_best_ke,col_trk_score],
    cuts=best_3D["cuts"],
    cut_sign=["<", "<" , "<"], 
)

ConfusionMatricesUtils.plot_confusion_matrix(
    cm = cm,
    x_labels = ["e/photon",r"$\mu/\pi$"],
    y_labels = ["e/photon",r"$\mu/\pi$"],
    cmap=sunset_cmap
)

print("2D no meandEdx")
cm = ConfusionMatricesUtils.build_confusion_matrix_from_cuts(
    signal_df=signal_df,
    bkg_df=bkg_df,
    cols=[col_mean_dEdx, col_best_ke,col_trk_score],
    cuts=[10,45, 0.62],
    cut_sign=["<", "<" , "<"], 
)


fig = ConfusionMatricesUtils.plot_confusion_matrix(
    cm = cm,
    x_labels = ["e/photon",r"$\mu/\pi$"],
    y_labels = ["e/photon",r"$\mu/\pi$"],
    cmap=sunset_cmap
)
fig.savefig(file_dir + "/confusion_matrix_michel_removal.png", dpi=300)
plt.show()




In [ ]:
# Split signal and background
cut_mask = (
    (evt_df.slc.cut.nu_score == True) & (evt_df.slc.cut.t0 == True) 
    & (evt_df.slc.cut.track == True) & (evt_df.slc.cut.inside_FV == True) 
    & (evt_df.slc.cut.MIP_candidates == True) & (evt_df.slc.cut.shower == True)
    & (evt_df.slc.cut.angle == True) & (evt_df.slc.cut.obvious_cosmic == True)
)

bkg_df = evt_df[
    cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

signal_df = evt_df[
    cut_mask & 
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 11) | (abs(evt_df.pfp.trk.truth.p.pdg) == 22)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

# -------------------------
# Columns
# -------------------------
col_chi2_mu = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p  = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_len  = ('pfp', 'trk', 'len', '', '', '')

col_num_hits  = ('pfp', 'max_daughter_hits', '', '', '', '')
col_mean_dEdx  = ('pfp', 'trk', 'mean_dEdx', '', '', '')
col_best_ke  = ('pfp', 'trk', 'calo', 'best', 'ke', '')
col_trk_score  = ('pfp', 'trackScore', '', '', '', '')


# -------------------------
# Remove rows with NaN or <=0 in the columns of interest
# -------------------------
signal_df = signal_df[(signal_df[col_chi2_mu] > 0) & (signal_df[col_chi2_p] > 0)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] > 0) & (bkg_df[col_chi2_p] > 0)]
signal_df = signal_df[(signal_df[col_chi2_mu] < 20) & (signal_df[col_chi2_p] > 85)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] < 20) & (bkg_df[col_chi2_p] > 85)]
signal_df = signal_df[(signal_df[col_num_hits] == 0)]
bkg_df    = bkg_df[bkg_df[col_num_hits] ==0]
signal_df = signal_df[(signal_df[col_len] > 10)]
bkg_df    = bkg_df[bkg_df[col_len] > 10]

# -------------------------
# Remove rows with NaN or <=0 in the columns of interest
# -------------------------
signal_df = signal_df[(signal_df[col_trk_score] < 0.6) & (signal_df[col_best_ke] < 45)]
bkg_df    = bkg_df[(bkg_df[col_trk_score] < 0.6) & (bkg_df[col_best_ke] < 45)]

df_opt_len, best_len, fig = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column= col_mean_dEdx,
    cut_type="<",
    xlabel="dEdx [MeV/cm]",
    title="pfp optimization",
    signal_name=r"$\gamma/e$",
    bkg_name=r"$\mu/\pi$",
    xlim=(0, 10),
    nbins=51,
    legend_loc="upper right",
    cut_unit="[MeV/cm]",
    normalize_hist = False,
    min_pur=0.0 # <-- new parameter
)